# PaliGemma-3B H3 mechanism: do the T-only heads read the note? (D80)

Owner: **@SohamRShinde**. Design and reading rule pre-registered in
`docs/DECISIONS.md` **D80** (D74, Qwen's H3, ported to PaliGemma) before this
notebook existed.

PaliGemma's H2 found the T attack runs largely through the **4 T-only heads**
(L20H7, L22H4, L22H5, L23H7; ablating them alone defeats it in 42.1% of
objects). This measures, at the last token, for **every head of all 26 layers**:

| measure | question |
|---|---|
| **attention mass on the note's image tokens** | do they look at the note? (test **A**) |
| **direct effect** (logit lens on the head's own output) | do they write the note's colour? (test **W**) |
| attention mass on the object's tokens, all image tokens | where else do they look? |

Each value is compared with **D73's 100 layer-matched random head sets**
(`cross_ablation.build_plan`, as PaliGemma's H2), by `h3.analyse_h3` unchanged.

**Runs per object.** T (302 objects): incongruent and neutral notes in 3
layouts (`on_object_small` is discovery's; `on_object_medium` and
`beside_object` are plan 3.7's spatial analysis, the first time PaliGemma
sees them), plus the original image with no note. P (121 objects): Prior and
Visual prompts on the counterfactual image. 2,356 runs.

**What differs from Qwen's H3 (D80):** full forward passes (PaliGemma's prefix
attends both ways); eager attention; Gemma-2's post-attention norm in the
direct effect; a pixel-patch check of the image-token order instead of M-RoPE's.
Section 3 stops the notebook if any measurement check fails.

**Runtime ~1.5-2.5 h estimated; section 3 measures it.** Checkpointed per
object (D44). To resume: attach the previous version's output and run again.
It can run **at the same time as `pg_v_patching`**: separate output folder.

**Untested on the GPU.** Settings: GPU T4, Internet on, `HF_TOKEN` ticked.
Run as **Save Version -> Save & Run All**.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/SohamRShinde/VLM-Overlap-RP"
REPO_DIR = "/kaggle/working/research_paper"

import importlib, os, subprocess, sys

if REPO_URL:
    if os.path.exists(REPO_DIR):
        # Always update an existing checkout: a stale clone silently runs old
        # code, and the failure shows up later as a confusing ImportError.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "-q", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# Drop any already-imported package modules: Python caches them, so after an
# update the kernel would keep serving the old file.
for _m in [m for m in list(sys.modules) if m.startswith("circuit_overlap")]:
    del sys.modules[_m]
importlib.invalidate_caches()

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("repo at", COMMIT)   # record this in RESULTS.md with any number from this run

# Fail here, loudly, if the checkout is older than this notebook expects.
import importlib.util as _ilu
_need = {}
for _mod, _names in {
    "circuit_overlap.pg_h3": ["FIRST_LAYER_PG", "PROBE_CELL", "measure_run_pg",
                              "check_attention_reconstructs_heads_full",
                              "check_direct_effect_formula_gemma", "check_patch_order"],
    "circuit_overlap.h3": ["SMALL_INC", "SMALL_NEU", "token_sets", "capture_last_token_attention",
                           "analyse_h3"],
    "circuit_overlap.paligemma": ["CHECKPOINT", "load_paligemma", "build_inputs", "text_geometry",
                                  "FullForwardRunner", "square_image_block", "quiet_processor_warning"],
    "circuit_overlap.cross_ablation": ["build_plan"],
    "circuit_overlap.notes": ["object_mask_from_pair"],
    "circuit_overlap.t_patching": ["render_t_pair", "LAYOUT", "load_patching_records",
                                   "is_t_conflicting"],
    "circuit_overlap.replication": ["lietzow_rows", "run_cached"],
    "circuit_overlap.t_screen": ["PRIOR_PROMPT", "VISUAL_PROMPT", "color_first_token_ids",
                                 "load_notes_config", "native_bf16"],
    "circuit_overlap.provenance": ["result_envelope", "write_result"],
}.items():
    if _ilu.find_spec(_mod) is None:
        _need[_mod] = ["module"]
        continue
    _m = importlib.import_module(_mod)
    _gone = [n for n in _names if not hasattr(_m, n)]
    if _gone:
        _need[_mod] = _gone
for _f in ("results/pg_replication_patching_paligemma2_3b.json",
           "results/pg_t_patching_paligemma2_3b.json",
           "results/pg_t_patching_records_paligemma2_3b.jsonl.gz",
           "scripts/analyze_pg_h3.py"):
    if not os.path.exists(f"{REPO_DIR}/{_f}"):
        _need[_f] = ["file"]
import inspect as _insp
if not _need:   # additions the name check cannot see
    from circuit_overlap.replication import run_cached as _rc
    from circuit_overlap.paligemma import load_paligemma as _lp
    for _what, _ok in (("run_cached(budget_s=...)", "budget_s" in _insp.signature(_rc).parameters),
                       ("load_paligemma(attn_implementation=...)",
                        "attn_implementation" in _insp.signature(_lp).parameters)):
        if not _ok:
            _need[_what] = ["parameter"]
if _need:
    raise RuntimeError(f"checkout {COMMIT} is missing {_need}. "
                       "Push the local commits to origin/main, then re-run this cell.")
print("code is current for this notebook")

In [ ]:
# Use one GPU. On "GPU T4 x2", device_map="auto" splits the model across both
# cards, which only adds transfer time for a 3B model. This must run before
# torch is first imported, so restart the kernel if torch is already loaded.
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# bitsandbytes is required for the 4-bit load (D10). Installing it into a
# kernel that already imported transformers does nothing: restart and re-run.
!pip install -q -U "transformers>=4.49" accelerate datasets inflect "bitsandbytes>=0.46.1"
import torch, transformers
from transformers.utils import is_bitsandbytes_available
from circuit_overlap.t_screen import native_bf16
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
# Not torch.cuda.is_bf16_supported(): it says True on a T4 (emulation).
print("bitsandbytes available:", is_bitsandbytes_available(),
      "| native bf16:", native_bf16(torch.cuda.get_device_capability(0)))

from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    # Secrets are attached PER NOTEBOOK: one on your account is not enough.
    raise RuntimeError(
        "HF_TOKEN is not attached to this notebook. In the editor: Add-ons -> "
        "Secrets -> tick the checkbox next to HF_TOKEN (exact label), then "
        "restart the session and run from cell 1. It must be ticked before "
        "Save & Run All, or the background run fails here."
    ) from exc

## 2. Model with eager attention (D80 (2))

sdpa returns no attention weights. Note that transformers' sdpa path skips
Gemma-2's attention-logit soft-capping and the eager path applies it, so
logits can move a little relative to discovery; section 3 measures that on
the first object and every object records it (D80 (7)).

In [ ]:
import json
from circuit_overlap.paligemma import CHECKPOINT, load_paligemma, quiet_processor_warning, text_geometry
from circuit_overlap.t_screen import PRIOR_PROMPT, VISUAL_PROMPT

quiet_processor_warning()   # the per-call `<image>` nag; section 3 checks the input itself

model, processor = load_paligemma(attn_implementation="eager")
COMPUTE_DTYPE = str(model.config.quantization_config.bnb_4bit_compute_dtype).replace("torch.", "")
ATTN = getattr(model.config.text_config, "_attn_implementation", None) or getattr(model.config, "_attn_implementation", "?")
print(CHECKPOINT, "| compute dtype:", COMPUTE_DTYPE, "| attention:", ATTN)
assert ATTN == "eager", f"attention implementation is {ATTN!r}, not eager: no weights to measure"
print("devices used:", sorted({str(d) for d in getattr(model, "hf_device_map", {"": model.device}).values()}))

geo = text_geometry(model)          # raises if o_proj's width != n_heads x head_dim
n_layers, n_heads, head_dim = geo["n_layers"], geo["n_heads"], geo["head_dim"]
print(geo)
assert (n_layers, n_heads) == (26, 8), "not the geometry P and T were discovered on"
assert hasattr(model.model.language_model.layers[0], "post_attention_layernorm"), "not a Gemma-2 layer (D80 (5))"

## 3. The plan, the objects, and the checks of the measurement (D80 (6))

The head sets and draws are PaliGemma's H2 plan. **The notebook stops if any
check fails:**

1. the image tokens are one 32 x 32 block, and changing one pixel patch at
   (row 5, column 21) changes vision-tower token 5 x 32 + 21 the most
   (row-major; a transposed grid would move token 21 x 32 + 5);
2. each head's attention weights sum to 1;
3. weights x values (same pass) **rebuild each head's `o_proj` input**;
4. the logit lens reproduces the model's logit difference, and the per-head
   contributions through `post_attention_layernorm` sum to its output;
5. the first object's eager answers equal discovery's.

It also saves `check_tokens.png` (note tokens red, object tokens blue on the
model's grid): look at it before trusting the masks.

In [ ]:
import glob, gzip, inflect, math, pathlib, shutil, time
from collections import Counter
import numpy as np
import torch
from PIL import Image
from IPython.display import display
from datasets import load_dataset
from circuit_overlap.paligemma import FullForwardRunner, build_inputs as _pg_inputs, square_image_block
from circuit_overlap.replication import lietzow_rows, run_cached
from circuit_overlap.t_screen import color_first_token_ids, load_notes_config
from circuit_overlap.t_patching import LAYOUT, render_t_pair, load_patching_records, is_t_conflicting
from circuit_overlap.cross_ablation import build_plan
from circuit_overlap.notes import object_mask_from_pair
from circuit_overlap.h3 import SMALL_INC, SMALL_NEU, token_sets, capture_last_token_attention
from circuit_overlap.pg_h3 import (FIRST_LAYER_PG, PROBE_CELL, measure_run_pg, check_patch_order,
                                   check_attention_reconstructs_heads_full,
                                   check_direct_effect_formula_gemma)

WORK = pathlib.Path("/kaggle/working/pg_h3_attention")
CKPT = WORK / "ckpt"            # checkpoints: NOT results
CKPT.mkdir(parents=True, exist_ok=True)
# Resume: bring back ckpt/ from an earlier run attached as an input. Found
# automatically; set RESUME_FROM to the folder holding h3_T.jsonl only if the
# search picks the wrong one.
RESUME_FROM = None
if RESUME_FROM is None:
    hits = sorted(glob.glob("/kaggle/input/**/h3_T.jsonl", recursive=True))
    RESUME_FROM = str(pathlib.Path(hits[0]).parent) if hits else None
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, CKPT, dirs_exist_ok=True)
for name in ("h3_T.jsonl", "h3_P.jsonl"):
    n_lines = sum(1 for l in open(CKPT / name) if l.strip()) if (CKPT / name).exists() else 0
    print(f"{name}: {n_lines} objects already done" + (f" (from {RESUME_FROM})" if RESUME_FROM else ""))

ds = load_dataset("mgolov/Visual-Counterfact", split="color")
_eng = inflect.engine()
rows, dropped = lietzow_rows(ds, _eng)
row_by_i = {r["i"]: r for r in rows}
vocab = sorted({c for r in rows for c in r["correct"]} | {r["attack"] for r in rows})
color_ids, collisions = color_first_token_ids(processor.tokenizer, vocab)
ambiguous = set(collisions)
print(len(rows), "objects |", len(vocab), "colours | sharing a first token (D26):", collisions)

notes_cfg = load_notes_config(f"{REPO_DIR}/configs/notes.json")
specs = {s.name: s for s in notes_cfg["specs"]}
neutral_words = notes_cfg["neutral_words"]
LAYOUTS = ("on_object_small", "on_object_medium", "beside_object")
assert LAYOUT == LAYOUTS[0] and all(l in specs for l in LAYOUTS), sorted(specs)

p_res = json.load(open(f"{REPO_DIR}/results/pg_replication_patching_paligemma2_3b.json"))["result"]
t_res = json.load(open(f"{REPO_DIR}/results/pg_t_patching_paligemma2_3b.json"))["result"]
plan = build_plan(p_res, t_res, n_heads_per_layer=n_heads)
for k, v in plan["sets"].items():
    print(f"  {k:7s} ({len(v)}): {', '.join(v)}")
assert plan["sets"]["T_only"] == ["L20H7", "L22H4", "L22H5", "L23H7"], "not D80's T-only set"
LAYERS = list(range(FIRST_LAYER_PG, n_layers))      # D80 (3): every layer

p_disc = {r["i"]: r for r in p_res["records"] if r["status"] == "patched"}
t_disc = {r["i"]: r for r in load_patching_records(
    f"{REPO_DIR}/results/pg_t_patching_records_paligemma2_3b.jsonl.gz")}
todo = {"T": [row_by_i[i] for i in t_disc], "P": [row_by_i[i] for i in p_disc]}
print(f"T {len(todo['T'])} objects, P {len(todo['P'])} (D80: 302, 121)")
assert (len(todo["T"]), len(todo["P"])) == (302, 121)
IMAGE_TOKEN_ID = processor.image_token_id

def build_inputs(image, prompt):
    return _pg_inputs(processor, image, prompt, model.device)

def prepared(inputs):
    """A full-forward runner, and the image block's offset and grid (D80 (4))."""
    offset, side = square_image_block(inputs["input_ids"][0].tolist(), IMAGE_TOKEN_ID)
    return FullForwardRunner(model, inputs), offset, side, side

def answers(d):
    """(source answer, target answer) token ids from a discovery record (D74 (4))."""
    return int(d["next_token_idx_prior"]), int(d["next_token_idx_visual"])

def finish(out, d, src_run, tgt_run):
    """Status by D74 (2): same answers as discovery, everything finite."""
    a = (out["runs"][src_run]["top1"], out["runs"][tgt_run]["top1"])
    ref = [d["prior_token_logit_prior_run"], d["visual_token_logit_prior_run"],
           d["prior_token_logit_visual_run"], d["visual_token_logit_visual_run"]]
    ours = out["runs"][src_run]["pair"] + out["runs"][tgt_run]["pair"]
    out["discovery_pair_diff"] = round(max(abs(x - y) for x, y in zip(ours, ref)), 4)
    if a != answers(d):
        out.update(status="drift", discovery=list(answers(d)))
    elif not all(m["finite"] for m in out["runs"].values()):
        out["status"] = "numerics_failed"                     # plan risk 7
    else:
        out["status"] = "ok"
    return out

VERIFY = {"compute_dtype": COMPUTE_DTYPE, "attention": ATTN, "geometry": geo}
r0 = todo["T"][0]
d0 = t_disc[r0["i"]]
tok0 = answers(d0)
row0 = ds[r0["i"]]
orig0, cf0 = row0["original_image"].convert("RGB"), row0["counterfact_image"].convert("RGB")
pr0 = render_t_pair(orig0, cf0, r0["attack"], neutral_words, specs[LAYOUT])
inp0 = build_inputs(pr0["incongruent"], VISUAL_PROMPT.format(object=r0["name"]))

# [VERIFY] 1. one square block, row-major by a pixel-patch probe.
runner0, offset0, side0, _ = prepared(inp0)
VERIFY["image_block"] = {"offset": offset0, "side": side0, "n_tokens": int(inp0["input_ids"].shape[1])}
assert side0 == 32, VERIFY["image_block"]
VERIFY["patch_order"] = check_patch_order(model, inp0["pixel_values"], side0, PROBE_CELL,
                                          dtype=getattr(torch, COMPUTE_DTYPE))
print("image block:", VERIFY["image_block"], "| patch order:", VERIFY["patch_order"])
# [VERIFY] 2. weights sum to 1 over the keys.
with capture_last_token_attention(model, LAYERS) as w0:
    runner0.logits()
VERIFY["max_weight_sum_error"] = max(float((w0[l].sum(-1) - 1).abs().max()) for l in LAYERS)
assert VERIFY["max_weight_sum_error"] < 1e-2, VERIFY
del w0
# [VERIFY] 3. weights x values rebuild every head's o_proj input slice.
VERIFY["reconstruction_rel_error"] = check_attention_reconstructs_heads_full(model, runner0, LAYERS, head_dim)
# [VERIFY] 4. the logit lens reproduces the model; per-head contributions sum to the norm's output.
VERIFY["direct_effect"] = check_direct_effect_formula_gemma(model, runner0, *tok0, layer=22, head_dim=head_dim)

# [VERIFY] 5. the measurement itself, and agreement with discovery's answers.
sets0 = token_sets(pr0["note_mask"], object_mask_from_pair(orig0, cf0), side0, side0, offset0)
t0_ = time.time()
m0 = measure_run_pg(model, runner0, sets0, *tok0, LAYERS, head_dim)
torch.cuda.synchronize()
run_s = time.time() - t0_
VERIFY["first_object"] = {"name": r0["name"], "n_tokens": m0["n_tokens"], "pair": m0["pair"],
                          "discovery_pair": [d0["prior_token_logit_prior_run"], d0["visual_token_logit_prior_run"]]}
print("first object:", VERIFY["first_object"])
assert m0["top1"] == tok0[0], f"{r0['name']}: eager answer {m0['top1']} vs discovery {tok0[0]}"

# Look at this picture: note tokens red, object tokens blue, on the model's grid.
patch_px = 448 // side0
img = pr0["incongruent"].resize((side0 * patch_px, side0 * patch_px))
tint = np.zeros((side0, side0, 3), np.uint8)
for k in sets0["note"] - offset0:
    tint[k // side0, k % side0] = (255, 0, 0)
for k in sets0["object"] - offset0:
    tint[k // side0, k % side0] = (0, 0, 255)
overlay = Image.fromarray(tint).resize(img.size, Image.NEAREST)
Image.blend(img, overlay, 0.35).save(WORK / "check_tokens.png")
display(Image.open(WORK / "check_tokens.png"))

# [VERIFY] the same checks on a P object (no note; Prior prompt).
r1 = todo["P"][0]
cf1 = ds[r1["i"]]["counterfact_image"].convert("RGB")
runner1, _, _, _ = prepared(build_inputs(cf1, PRIOR_PROMPT.format(a_object=_eng.a(r1["name"]))))
VERIFY["reconstruction_rel_error_P"] = check_attention_reconstructs_heads_full(model, runner1, LAYERS, head_dim)
VERIFY["direct_effect_P"] = check_direct_effect_formula_gemma(model, runner1, *answers(p_disc[r1["i"]]),
                                                              layer=15, head_dim=head_dim)
print("checks passed:", {k: v for k, v in VERIFY.items() if k not in ("first_object", "geometry")})

runs = 7 * len(todo["T"]) + 2 * len(todo["P"])
VERIFY["run_s"] = run_s
print(f"one measured run {run_s:.2f} s x {runs} runs -> ~{runs * run_s / 3600:.1f} h "
      "(plus rendering)")
del runner0, runner1

## 4. T objects

Per object: 3 layouts x (incongruent, neutral), then the original image with
no note, whose note mass is read on the `on_object_small` footprint. Each
layout's fool status is recorded on this path (`is_t_conflicting`). An object
whose `on_object_small` answers differ from discovery is kept in the file but
marked `drift` and not analysed (D80 (7)).

In [ ]:
SESSION_BUDGET_H = 11.0   # stop starting new objects after this many hours (Kaggle stops at 12 h)
DEADLINE = time.time() + SESSION_BUDGET_H * 3600

def left():
    return max(DEADLINE - time.time(), 0)

def t_object(r):
    d = t_disc[r["i"]]
    tok = answers(d)                       # (note colour, real colour)
    row = ds[r["i"]]
    original, cf = row["original_image"].convert("RGB"), row["counterfact_image"].convert("RGB")
    obj_mask = object_mask_from_pair(original, cf)
    prompt = VISUAL_PROMPT.format(object=r["name"])
    out = {"condition": "T", "name": r["name"], "tokens": list(tok), "runs": {}, "fooled": {}}
    try:
        for layout in LAYOUTS:
            pr = render_t_pair(original, cf, r["attack"], neutral_words, specs[layout])
            if layout == LAYOUT:
                small_mask = pr["note_mask"]
            top1 = {}
            for variant in ("incongruent", "neutral"):
                runner, offset, g_rows, g_cols = prepared(build_inputs(pr[variant], prompt))
                sets = token_sets(pr["note_mask"], obj_mask, g_rows, g_cols, offset)
                out["runs"][f"{layout}/{variant}"] = measure_run_pg(model, runner, sets, *tok, LAYERS, head_dim)
                top1[variant] = out["runs"][f"{layout}/{variant}"]["top1"]
            out["fooled"][layout] = is_t_conflicting(r, top1["neutral"], top1["incongruent"],
                                                     color_ids, ambiguous)
        runner, offset, g_rows, g_cols = prepared(build_inputs(original, prompt))
        sets = token_sets(small_mask, obj_mask, g_rows, g_cols, offset)
        out["runs"]["clean"] = measure_run_pg(model, runner, sets, *tok, LAYERS, head_dim)
    except ValueError as exc:              # square_image_block refused this input
        return {**out, "status": "layout_check_failed", "error": str(exc), "runs": {}}
    return finish(out, d, SMALL_INC, SMALL_NEU)

h3 = {"T": run_cached(CKPT / "h3_T.jsonl", todo["T"], t_object, progress_every=10, budget_s=left())}
print("T", f"{len(h3['T'])}/{len(todo['T'])} done", dict(Counter(r["status"] for r in h3["T"])))

## 5. P objects

Prior and Visual prompts on the counterfactual image, as in discovery. D74
(8)'s routing test is **descriptive here** (D80 (8)): Lietzow report that
PaliGemma's heads do not shift attention between prompts.

In [ ]:
def p_object(r):
    d = p_disc[r["i"]]
    tok = answers(d)                       # (prior answer, image colour)
    row = ds[r["i"]]
    cf = row["counterfact_image"].convert("RGB")
    obj_mask = object_mask_from_pair(row["original_image"].convert("RGB"), cf)
    out = {"condition": "P", "name": r["name"], "tokens": list(tok), "runs": {}}
    prompts = {"prior": PRIOR_PROMPT.format(a_object=_eng.a(r["name"])),
               "visual": VISUAL_PROMPT.format(object=r["name"])}
    try:
        for run, prompt in prompts.items():
            runner, offset, g_rows, g_cols = prepared(build_inputs(cf, prompt))
            sets = token_sets(None, obj_mask, g_rows, g_cols, offset)
            out["runs"][run] = measure_run_pg(model, runner, sets, *tok, LAYERS, head_dim, with_note=False)
    except ValueError as exc:
        return {**out, "status": "layout_check_failed", "error": str(exc), "runs": {}}
    return finish(out, d, "prior", "visual")

h3["P"] = run_cached(CKPT / "h3_P.jsonl", todo["P"], p_object, progress_every=20, budget_s=left())
print("P", f"{len(h3['P'])}/{len(todo['P'])} done", dict(Counter(r["status"] for r in h3["P"])))

## 6. Save (after every session)

The numbers printed here are **preliminary**: the reported ones come from
`scripts/analyze_pg_h3.py` on the two files saved below.

In [ ]:
from circuit_overlap.h3 import analyse_h3
from circuit_overlap.t_screen import native_bf16
from circuit_overlap.provenance import result_envelope, write_result

complete = len(h3["T"]) == len(todo["T"]) and len(h3["P"]) == len(todo["P"])
if complete:
    prelim = analyse_h3(plan, h3, n_boot=200, first_layer=FIRST_LAYER_PG)
    for name, t in prelim["primary"].items():
        print(f"PRELIMINARY {name}: {t['observed']:+.4f} vs random {t['null_mean']:+.4f}, "
              f"p = {t['p']:.4f} -> {t['wording']}")
    print("PRELIMINARY mechanism:", prelim["mechanism"])
else:
    print("not complete: no preliminary analysis; attach this version's output and run again")

summary = {
    "plan": plan,
    "complete": complete,
    "status": {c: dict(Counter(r["status"] for r in h3[c])) for c in h3},
    "verify": VERIFY,
    "layers_stored": [LAYERS[0], LAYERS[-1]],
    "records_file": "pg_h3_attention_records_paligemma2_3b.jsonl.gz",
}
print(write_result(WORK / "pg_h3_attention_paligemma2_3b.json", result_envelope(
    "pg_h3_attention", summary, model="PaliGemma2-3B-mix-448", checkpoint=CHECKPOINT, seed=0,
    prompt=f"T: visual={VISUAL_PROMPT!r}, incongruent vs neutral note, 3 layouts, and no note; "
           f"P: prior={PRIOR_PROMPT!r} vs visual, counterfactual image",
    n=sum(len(v) for v in h3.values()), repo_dir=REPO_DIR, decision="D80",
    quantisation="4-bit nf4, double quant", compute_dtype=COMPUTE_DTYPE,
    gpu=torch.cuda.get_device_name(0), image_tokens=side0 * side0, layouts=list(LAYOUTS),
    attention="eager; last-token query; full forward passes")))
with gzip.open(WORK / "pg_h3_attention_records_paligemma2_3b.jsonl.gz", "wt") as f:
    for cond in ("T", "P"):
        for r in h3[cond]:
            f.write(json.dumps(r) + "\n")
print("\nDownload ONLY these two into results/ (ckpt/ and check_tokens.png are not results):")
for name in ("pg_h3_attention_paligemma2_3b.json", "pg_h3_attention_records_paligemma2_3b.jsonl.gz"):
    print(f"  {name:48s} {(WORK / name).stat().st_size / 1e6:6.2f} MB")